# RET-02 — Verification Notebook

Imports the exact same `src/` modules used by `app.py` and runs sanity
checks across the sanitizer, model manager, inference engine, i18n catalog,
and edge-case routing. Prints **ALL CHECKS PASSED** at the end if every
check succeeds.


In [1]:
import os, sys

# Make sure `src/` (this notebook's own directory) is importable regardless
# of the working directory Jupyter was launched from.
PROJECT_ROOT = os.path.dirname(os.path.abspath("__file__"))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd

from src.config import CFG
from src.data_generator import SyntheticReviewDataGenerator
from src.sanitizer import ReviewSanitizer
from src.model_manager import ModelManager, BASELINE_MODE, OFFLINE_MODE
from src.inference_engine import ProductReviewIntelligenceEngine, STATUS_AUTO, STATUS_HUMAN
from src.analytics import generate_executive_summary, responses_to_dataframe
from src.i18n import LANGUAGES, t, _TRANSLATIONS

failures = []

def check(name: str, condition: bool, detail: str = ""):
    status = "PASS" if condition else "FAIL"
    print(f"[{status}] {name}" + (f" — {detail}" if detail and not condition else ""))
    if not condition:
        failures.append(name)


## 1. Sanitizer — cleaning and edge-case classification

In [2]:
sanitizer = ReviewSanitizer(CFG.min_alpha_ratio, CFG.min_token_count)

check("clean_text handles None", sanitizer.clean_text(None) == "")
check("clean_text handles NaN", sanitizer.clean_text(float("nan")) == "")
check("clean_text handles empty string", sanitizer.clean_text("") == "")
check("clean_text strips HTML tags", "<br>" not in sanitizer.clean_text("<br>great product<br>"))
check("clean_text lowercases", sanitizer.clean_text("GREAT PRODUCT") == "great product")

check("classify_edge_case: empty text -> 'empty'", sanitizer.classify_edge_case("") == "empty")
check("classify_edge_case: 'ok' -> 'ultra_short'", sanitizer.classify_edge_case(sanitizer.clean_text("ok")) == "ultra_short")
check("classify_edge_case: '???' -> 'empty' or 'gibberish'",
      sanitizer.classify_edge_case(sanitizer.clean_text("???")) in ("empty", "gibberish"))
check("classify_edge_case: normal sentence -> 'ok'",
      sanitizer.classify_edge_case(sanitizer.clean_text("This product works great and arrived on time")) == "ok")


[PASS] clean_text handles None
[PASS] clean_text handles NaN
[PASS] clean_text handles empty string
[PASS] clean_text strips HTML tags
[PASS] clean_text lowercases
[PASS] classify_edge_case: empty text -> 'empty'
[PASS] classify_edge_case: 'ok' -> 'ultra_short'
[PASS] classify_edge_case: '???' -> 'empty' or 'gibberish'
[PASS] classify_edge_case: normal sentence -> 'ok'


## 2. Synthetic data generator

In [3]:
gen = SyntheticReviewDataGenerator(n_samples=200, seed=CFG.seed)
df_demo = gen.generate()

expected_cols = {"review_id", "product_id", "category", "rating", "review_title",
                  "review_body", "issue_category", "sentiment", "actionability_score"}
check("generate() returns expected columns", expected_cols.issubset(set(df_demo.columns)))
check("generate() returns requested row count", len(df_demo) == 200)
check("ratings are within [1, 5]", df_demo["rating"].between(1, 5).all())
check("actionability_score within [0, 1]", df_demo["actionability_score"].between(0, 1).all())
check("sentiment values are within controlled vocabulary",
      set(df_demo["sentiment"].unique()).issubset(set(CFG.sentiment_classes)))


[PASS] generate() returns expected columns
[PASS] generate() returns requested row count
[PASS] ratings are within [1, 5]
[PASS] actionability_score within [0, 1]
[PASS] sentiment values are within controlled vocabulary


## 3. Model loading / training (ModelManager)

In [4]:
model_manager = ModelManager().build()

check("ModelManager builds successfully", model_manager.model is not None)
check("Active mode is baseline or offline", model_manager.mode in (BASELINE_MODE, OFFLINE_MODE))
check("issue_classes populated", len(model_manager.issue_classes) == len(CFG.issue_categories))
check("sentiment_classes populated", len(model_manager.sentiment_classes) == len(CFG.sentiment_classes))
print(f"Active backbone mode: {model_manager.mode} | training rows: {model_manager.training_rows}")


[PASS] ModelManager builds successfully
[PASS] Active mode is baseline or offline
[PASS] issue_classes populated
[PASS] sentiment_classes populated
Active backbone mode: baseline | training rows: 684


## 4. Inference on sample texts

In [5]:
sample_clean = [sanitizer.clean_text(t_) for t_ in
                ["Great product, fast shipping, would buy again!",
                 "The item arrived broken and support never responded"]]
bundle = model_manager.predict(sample_clean)

check("predict() returns issue_probs with correct shape",
      bundle.issue_probs.shape == (2, len(model_manager.issue_classes)))
check("predict() returns sentiment_probs with correct shape",
      bundle.sentiment_probs.shape == (2, len(model_manager.sentiment_classes)))
check("predict() returns actionability of correct length", len(bundle.actionability) == 2)
check("actionability predictions are within [0, 1]",
      bool(np.all((bundle.actionability >= 0) & (bundle.actionability <= 1))))


[PASS] predict() returns issue_probs with correct shape
[PASS] predict() returns sentiment_probs with correct shape
[PASS] predict() returns actionability of correct length
[PASS] actionability predictions are within [0, 1]


## 5. Inference engine — confidence routing & edge cases

In [6]:
engine = ProductReviewIntelligenceEngine(model_manager, sanitizer, CFG.confidence_threshold)

edge_case_tests = [
    ("", "empty"),
    (None, "empty"),
    ("???", None),          # 'empty' or 'gibberish' depending on punctuation-only handling
    ("ok", "ultra_short"),
]
for text, expected in edge_case_tests:
    resp = engine.analyze(text, review_id="EDGE-TEST")
    ok = (expected is None) or (resp.edge_case_flag == expected)
    check(f"analyze({text!r}) edge_case_flag as expected", ok, detail=f"got {resp.edge_case_flag}")
    check(f"analyze({text!r}) forces HUMAN_REVIEW_REQUIRED", resp.status == STATUS_HUMAN)

normal_resp = engine.analyze(
    "The package arrived crushed and the item inside was defective, does not turn on at all.",
    rating=1, review_id="NORMAL-TEST",
)
check("analyze() on a normal review returns edge_case_flag='ok'", normal_resp.edge_case_flag == "ok")
check("analyze() overall_confidence within [0, 1]", 0.0 <= normal_resp.overall_confidence <= 1.0)
check("analyze() status is a valid routing value", normal_resp.status in (STATUS_AUTO, STATUS_HUMAN))
check("analyze() never raises on malformed input (None text)",
      engine.analyze(None, review_id="NONE-TEST").status == STATUS_HUMAN)


[PASS] analyze('') edge_case_flag as expected
[PASS] analyze('') forces HUMAN_REVIEW_REQUIRED
[PASS] analyze(None) edge_case_flag as expected
[PASS] analyze(None) forces HUMAN_REVIEW_REQUIRED
[PASS] analyze('???') edge_case_flag as expected
[PASS] analyze('???') forces HUMAN_REVIEW_REQUIRED
[PASS] analyze('ok') edge_case_flag as expected
[PASS] analyze('ok') forces HUMAN_REVIEW_REQUIRED
[PASS] analyze() on a normal review returns edge_case_flag='ok'
[PASS] analyze() overall_confidence within [0, 1]
[PASS] analyze() status is a valid routing value
[PASS] analyze() never raises on malformed input (None text)


## 6. Batch analysis & executive summary aggregation

In [7]:
batch_records = [
    {"review_text": "Great quality, fast shipping, would definitely buy again!", "rating": 5, "review_id": "B1"},
    {"review_text": "Item broke after two days, totally defective", "rating": 1, "review_id": "B2"},
    {"review_text": "ok", "rating": 3, "review_id": "B3"},
    {"review_text": "", "rating": None, "review_id": "B4"},
]
responses = engine.batch_analyze(batch_records)
check("batch_analyze() returns one response per input", len(responses) == len(batch_records))

results_df = responses_to_dataframe(responses)
check("responses_to_dataframe() row count matches input", len(results_df) == len(batch_records))
check("responses_to_dataframe() has expected columns",
      {"review_id", "sentiment", "status", "actionability_score"}.issubset(set(results_df.columns)))

summary = generate_executive_summary(responses)
check("executive summary total_reviews matches batch size", summary.total_reviews == len(batch_records))
check("executive summary auto + human == total",
      summary.auto_processed + summary.human_review_required == summary.total_reviews)
check("executive summary NSS within [-100, 100]", -100.0 <= summary.net_sentiment_score <= 100.0)

empty_summary = generate_executive_summary([])
check("executive summary handles empty input without raising", empty_summary.total_reviews == 0)


[PASS] batch_analyze() returns one response per input
[PASS] responses_to_dataframe() row count matches input
[PASS] responses_to_dataframe() has expected columns
[PASS] executive summary total_reviews matches batch size
[PASS] executive summary auto + human == total
[PASS] executive summary NSS within [-100, 100]
[PASS] executive summary handles empty input without raising


## 7. i18n — trilingual coverage

In [8]:
check("LANGUAGES contains en/uz/ru", set(LANGUAGES.keys()) == {"en", "uz", "ru"})

en_keys = set(_TRANSLATIONS["en"].keys())
for lang in ["uz", "ru"]:
    lang_keys = set(_TRANSLATIONS[lang].keys())
    check(f"'{lang}' translation catalog has no missing keys vs 'en'", en_keys - lang_keys == set(),
          detail=str(en_keys - lang_keys))
    check(f"'{lang}' translation catalog has no extra keys vs 'en'", lang_keys - en_keys == set(),
          detail=str(lang_keys - en_keys))

check("t() falls back gracefully for an unknown key", t("__nonexistent_key__", "en") == "__nonexistent_key__")
check("t() returns non-empty strings for a sample of real keys",
      all(len(t(k, lang)) > 0 for k in ["app_title", "analyze_btn", "no_review_warning"] for lang in LANGUAGES))


[PASS] LANGUAGES contains en/uz/ru
[PASS] 'uz' translation catalog has no missing keys vs 'en'
[PASS] 'uz' translation catalog has no extra keys vs 'en'
[PASS] 'ru' translation catalog has no missing keys vs 'en'
[PASS] 'ru' translation catalog has no extra keys vs 'en'
[PASS] t() falls back gracefully for an unknown key
[PASS] t() returns non-empty strings for a sample of real keys


## 8. Final verdict

In [9]:
print("=" * 60)
if not failures:
    print("ALL CHECKS PASSED")
else:
    print(f"{len(failures)} CHECK(S) FAILED:")
    for f in failures:
        print(f"  - {f}")
    raise AssertionError(f"{len(failures)} verification check(s) failed")


ALL CHECKS PASSED
